# Phase E Pretrain Streaming (Kaggle Full Flow)

- Change class: `capability_expansion`
- Scope: 100M streaming pretrain (50M + 50M resume), Phase E gate freeze/diff, ARC probe, weight export
- Mandatory docs consulted:
  - Section 1: `docs/10`, `docs/01`, `docs/02`, `docs/03`, `docs/04`
  - Section 2: `docs/05`, `docs/06`, `docs/08`
  - Data policy: `docs/07`

Expected status rules:
- `Done`: 100M flow complete, S8 crash classes rebuilt, probe artifact exists, Phase E diff summary is `PASS`
- `Blocked`: any hard gate failure (`S1..S8`, `S8_h100_packet`, `PhaseEProbe`) or missing required artifacts


In [ ]:
import json
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

REPO_INPUT_DIR = Path('/kaggle/input/datasets/danielwuu/iris-main/packages')
WHEEL_DIR = Path('/kaggle/input/datasets/danielwuu/wheels/wheels/linux_cp312')
WORK_REPO = Path('/kaggle/working/IRIS')

ARTIFACT_ROOT = WORK_REPO / 'artifacts' / 'phase_e_pretrain_streaming'
PREFERRED_SNAPSHOT_ROOT = Path(os.environ.get('IRIS_SNAPSHOT_ROOT', '/kaggle/input/iris-pure-lm-snapshots'))
PREFERRED_TOKENIZER_PATH = Path(os.environ.get('IRIS_TOKENIZER_PATH', '/kaggle/input/iris-tokenizer'))
TOKENIZER_FALLBACK_ID = os.environ.get('IRIS_TOKENIZER_ID', 'gpt2')
ALLOW_HF_ONLINE_FALLBACK = True
ENABLE_BOOTSTRAP_LOCAL_SNAPSHOT_FALLBACK = True
ALLOW_SMOKE_CPU_FALLBACK = True
BOOTSTRAP_SNAPSHOT_ROOT = ARTIFACT_ROOT / 'bootstrap_local_snapshot'
BOOTSTRAP_TOKENIZER_DIR = ARTIFACT_ROOT / 'bootstrap_tokenizer'
BOOTSTRAP_SNAPSHOT_RECORDS_PER_SOURCE = 256
REQUIRE_GATE_PASS = True

if not WORK_REPO.exists():
    if not REPO_INPUT_DIR.exists():
        raise FileNotFoundError(f'Missing repo input dir: {REPO_INPUT_DIR}')
    shutil.copytree(REPO_INPUT_DIR, WORK_REPO)
    print('Copied repo to:', WORK_REPO)
else:
    print('Using existing repo at:', WORK_REPO)

WORK_REPO.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_REPO)

if PREFERRED_SNAPSHOT_ROOT.exists():
    STREAMING_MODE = 'local_snapshot'
    SNAPSHOT_ROOT: Path | None = PREFERRED_SNAPSHOT_ROOT
else:
    STREAMING_MODE = 'hf_online' if ALLOW_HF_ONLINE_FALLBACK else 'local_snapshot'
    SNAPSHOT_ROOT = None

PYTHON = Path(sys.executable)
ROOT = WORK_REPO


def run_shell(cmd: str, cwd: Path | None = None, ok: tuple[int, ...] = (0,), check: bool = True, env: dict | None = None):
    print('$', cmd)
    proc = subprocess.run(
        cmd,
        cwd=str(cwd or ROOT),
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
    )
    print(proc.stdout)
    if check and proc.returncode not in ok:
        raise RuntimeError(f'command failed (rc={proc.returncode}): {cmd}')
    return proc


def run_py(args: list[str], cwd: Path | None = None, ok: tuple[int, ...] = (0,), check: bool = True, env: dict | None = None):
    cmd = [str(PYTHON)] + [str(arg) for arg in args]
    print('$', ' '.join(shlex.quote(part) for part in cmd))
    proc = subprocess.run(
        cmd,
        cwd=str(cwd or ROOT),
        shell=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
    )
    print(proc.stdout)
    if check and proc.returncode not in ok:
        raise RuntimeError(f'command failed (rc={proc.returncode}): {" ".join(cmd)}')
    return proc


def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows: list[dict] = []
    for line in path.read_text(encoding='utf-8-sig').splitlines():
        line = line.strip()
        if not line:
            continue
        rows.append(json.loads(line))
    return rows


def _sample_text_for_source(source_id: str, index: int) -> str:
    idx = int(index)
    common = (
        f'Document {idx} keeps coherent language, stable syntax, semantic continuity, '
        f'and deterministic replay behavior for pretraining coverage. '
    )
    if source_id == 'open_web_math':
        return (
            common
            + f'In type theory we write \\Gamma \\vdash t_{idx} : T and reason with \\lambda terms, '
            + f'including \\sum_{{i=1}}^{{{(idx % 7) + 3}}} i and \\int_0^1 x^2 dx under bounded domains. '
            + 'Formal symbols remain explicit for math-signal filtering.'
        )
    if source_id == 'openstax_text':
        return (
            f'Algorithm: Step 1 parse input batch {idx}. Step 2 transform symbols with constraints. '
            f'Step 3 verify output and report diagnostics. Input: structured sequence {idx}. '
            'Output: validated result with explicit checks and bounded error analysis. '
            + common
        )
    if source_id == 'the_stack_code':
        mod_base = (idx % 13) + 17
        offset = idx % 9
        func_name = f'transform_grid_{idx}'
        code_stub = (
            f'def {func_name}(values):\n'
            '    result = []\n'
            f'    offset = {offset}\n'
            '    for i, value in enumerate(values):\n'
            f'        result.append((value + offset + i) % {mod_base})\n'
            '    return result\n'
        )
        return (
            code_stub
            + f'Python code sample {idx} emphasizes deterministic transformation, typed variables, and debug visibility. '
            + common
        )
    if source_id == 'redpajama_arxiv':
        return (
            common
            + 'Dependent type systems and lambda calculus support compositional proofs in type theory. '
            + f'We derive judgments with Gamma turnstile rules for lemma index {idx}. '
            + 'The manuscript section describes assumptions, methods, and evaluation criteria.'
        )
    if source_id == 'lean4_mathlib':
        lean_stub = (
            f'theorem sample_theorem_{idx} : True := by\n'
            '  trivial\n'
        )
        return (
            lean_stub
            + f'Formal Lean theorem sample {idx} with explicit proof script and stable structure. '
            + common
        )
    if source_id == 'redpajama_stackexchange':
        return (
            common
            + f'This answer for case {idx} provides practical debugging steps, constraints, and reproducible checks. '
            + 'It favors concise procedures and concrete diagnostics over vague advice.'
        )
    if source_id == 'pes2o_s2orc':
        return (
            common
            + f'The manuscript section {idx} details methods, assumptions, and evaluation criteria. '
            + 'It preserves structured academic phrasing and technical terminology.'
        )
    return (
        common
        + f'This document fragment {idx} preserves context, structure, and procedural details for robust training.'
    )


def _build_record(source_id: str, text_field: str, index: int) -> dict:
    rec = {text_field: _sample_text_for_source(source_id, index)}
    if source_id == 'pes2o_s2orc':
        rec['source'] = 's2orc'
    elif source_id == 'the_stack_code':
        rec['lang'] = 'python'
        rec['ext'] = '.py'
        rec['path'] = f'sample_{index}.py'
    elif source_id == 'lean4_mathlib':
        rec['type'] = 'theorem'
    elif source_id == 'redpajama_arxiv':
        rec['red_pajama_subset'] = 'arxiv'
        rec['domain'] = 'arxiv.org'
    elif source_id == 'redpajama_stackexchange':
        rec['red_pajama_subset'] = 'stackexchange'
        rec['domain'] = 'math.stackexchange.com'
    return rec


def bootstrap_local_snapshot(snapshot_root: Path, records_per_source: int = BOOTSTRAP_SNAPSHOT_RECORDS_PER_SOURCE):
    from iris.train.data.contracts import load_default_pure_lm_profile
    from iris.train.data.filters import prepare_clean_text

    profile_path = ROOT / 'src/iris/train/data/profiles/pure_lm_90_v1.json'
    if not profile_path.exists():
        raise FileNotFoundError(f'Pure LM profile not found: {profile_path}')
    profile = json.loads(profile_path.read_text(encoding='utf-8-sig'))
    sources = profile.get('sources', [])
    if not isinstance(sources, list) or not sources:
        raise RuntimeError('Invalid Pure LM profile: sources missing.')

    source_specs = load_default_pure_lm_profile().source_by_id

    shutil.rmtree(snapshot_root, ignore_errors=True)
    snapshot_root.mkdir(parents=True, exist_ok=True)

    created = []
    valid_counts: dict[str, int] = {}
    for source in sources:
        source_id = str(source.get('source_id', '')).strip()
        text_field = str(source.get('text_field', 'text')).strip() or 'text'
        if not source_id:
            continue

        source_dir = snapshot_root / source_id
        source_dir.mkdir(parents=True, exist_ok=True)
        out_path = source_dir / 'bootstrap.jsonl'

        valid_count = 0
        source_spec = source_specs.get(source_id)
        with out_path.open('w', encoding='utf-8') as handle:
            for idx in range(int(records_per_source)):
                row = _build_record(source_id, text_field, idx)
                handle.write(json.dumps(row, ensure_ascii=False) + '\n')
                if source_spec is not None and prepare_clean_text(source_spec, row):
                    valid_count += 1

        if valid_count <= 0:
            raise RuntimeError(
                f'Bootstrap snapshot source {source_id} has zero QA-passing records. '
                'Adjust generated text to satisfy filters/QA gate.'
            )

        valid_counts[source_id] = valid_count
        created.append(str(out_path))

    print('Bootstrap local snapshot ready:', snapshot_root)
    print('Bootstrap files count:', len(created))
    print('Bootstrap valid records by source:', valid_counts)
    return snapshot_root


def _validate_tokenizer_load(id_or_path: str) -> None:
    from transformers import AutoTokenizer

    AutoTokenizer.from_pretrained(
        id_or_path,
        use_fast=True,
        trust_remote_code=False,
    )


def ensure_bootstrap_tokenizer(tokenizer_dir: Path):
    config_path = tokenizer_dir / 'tokenizer_config.json'
    if config_path.exists():
        try:
            _validate_tokenizer_load(str(tokenizer_dir))
            return tokenizer_dir
        except Exception:
            shutil.rmtree(tokenizer_dir, ignore_errors=True)

    tokenizer_dir.mkdir(parents=True, exist_ok=True)

    # Use a standard BertTokenizerFast artifact so AutoTokenizer can reload it reliably.
    vocab_tokens = [
        '[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]',
        'This', 'sample', 'keeps', 'long', 'coherent', 'language', 'stable', 'syntax', 'semantic',
        'continuity', 'pretraining', 'coverage', 'deterministic', 'replay', 'behavior', 'Algorithm',
        'Step', 'Input', 'Output', 'Gamma', 'vdash', 'lambda', 'type', 'theory', 'proof', 'grid',
        'transform', 'result', 'debugging', 'pipeline', 'math', 'sequence', 'verify', 'constraints',
        '##ing', '##ed', '##ly', '##tion', '##s', ':', '.', ',', '(', ')', '[', ']', '{', '}', '+', '-',
        '/', '=', '\\', '\\Gamma', '\\vdash', '\\lambda', '\\sum', '\\int',
    ]
    vocab_path = tokenizer_dir / 'vocab.txt'
    vocab_path.write_text('\n'.join(vocab_tokens) + '\n', encoding='utf-8')

    from transformers import BertTokenizerFast

    tok = BertTokenizerFast(
        vocab_file=str(vocab_path),
        do_lower_case=False,
        unk_token='[UNK]',
        sep_token='[SEP]',
        pad_token='[PAD]',
        cls_token='[CLS]',
        mask_token='[MASK]',
    )
    tok.save_pretrained(str(tokenizer_dir))
    _validate_tokenizer_load(str(tokenizer_dir))
    return tokenizer_dir


if PREFERRED_TOKENIZER_PATH.exists():
    TOKENIZER_ID_OR_PATH = str(PREFERRED_TOKENIZER_PATH)
    _validate_tokenizer_load(TOKENIZER_ID_OR_PATH)
else:
    bootstrap_tok = ensure_bootstrap_tokenizer(BOOTSTRAP_TOKENIZER_DIR)
    TOKENIZER_ID_OR_PATH = str(bootstrap_tok)

print('python:', sys.version)
print('cwd:', ROOT)
print('repo input exists:', REPO_INPUT_DIR.exists(), REPO_INPUT_DIR)
print('wheel dir exists:', WHEEL_DIR.exists(), WHEEL_DIR)
print('preferred snapshot root:', PREFERRED_SNAPSHOT_ROOT)
print('effective streaming mode (initial):', STREAMING_MODE)
print('effective snapshot root (initial):', SNAPSHOT_ROOT)
print('preferred tokenizer path:', PREFERRED_TOKENIZER_PATH)
print('effective tokenizer-id-or-path:', TOKENIZER_ID_OR_PATH)


In [ ]:
import importlib.metadata as ilm


def wheel_version(path: Path) -> str:
    # wheel format: {name}-{version}-{python_tag}-...
    return path.name.split('-')[1]


def pkg_version(name: str) -> str | None:
    try:
        return ilm.version(name)
    except ilm.PackageNotFoundError:
        return None


if WHEEL_DIR.exists():
    run_shell(f"{PYTHON} -m pip install --no-index --find-links {WHEEL_DIR} --upgrade pip")

    jax_whls = sorted(WHEEL_DIR.glob('jax-*.whl'))
    jaxlib_whls = sorted(WHEEL_DIR.glob('jaxlib-*.whl'))
    if jax_whls and jaxlib_whls:
        jax_by_version = {wheel_version(p): p for p in jax_whls}
        jaxlib_by_version = {wheel_version(p): p for p in jaxlib_whls}
        common_versions = sorted(set(jax_by_version) & set(jaxlib_by_version))
        if not common_versions:
            raise RuntimeError('No common jax/jaxlib wheel version found in WHEEL_DIR.')

        target_version = common_versions[-1]
        plugin_by_version = {wheel_version(p): p for p in sorted(WHEEL_DIR.glob('jax_cuda12_plugin-*.whl'))}
        pjrt_by_version = {wheel_version(p): p for p in sorted(WHEEL_DIR.glob('jax_cuda12_pjrt-*.whl'))}

        wheels_to_install = [jax_by_version[target_version], jaxlib_by_version[target_version]]
        if target_version in plugin_by_version and target_version in pjrt_by_version:
            wheels_to_install.extend([plugin_by_version[target_version], pjrt_by_version[target_version]])
            print('Using matched jax/jaxlib/plugin/pjrt version:', target_version)
        else:
            print('Matched CUDA plugin/pjrt wheel not found for version', target_version)
            print('Uninstalling stale jax_cuda12_plugin/jax_cuda12_pjrt to avoid version mismatch errors.')
            run_shell(f"{PYTHON} -m pip uninstall -y jax_cuda12_plugin jax_cuda12_pjrt", ok=(0, 1))

        wheel_args = ' '.join(str(path) for path in wheels_to_install)
        run_shell(
            f"{PYTHON} -m pip install --no-index --find-links {WHEEL_DIR} "
            f"--upgrade --force-reinstall --no-deps {wheel_args}"
        )
    else:
        print('jax/jaxlib wheels not found in WHEEL_DIR; using package-name install fallback.')
        run_shell(
            f"{PYTHON} -m pip install --no-index --find-links {WHEEL_DIR} "
            "--upgrade --force-reinstall jax jaxlib"
        )

    run_shell(
        f"{PYTHON} -m pip install --no-index --find-links {WHEEL_DIR} "
        "--upgrade --force-reinstall numpy scipy ml-dtypes flax optax orbax-checkpoint datasets transformers"
    )
else:
    print('Wheel directory not found, skipping offline reinstall and using runtime packages.')

# Final guardrail: if plugin version still mismatches jaxlib, try to repair or uninstall stale plugins.
jaxlib_ver = pkg_version('jaxlib')
plugin_ver = pkg_version('jax_cuda12_plugin')
pjrt_ver = pkg_version('jax_cuda12_pjrt')

if jaxlib_ver and plugin_ver and plugin_ver != jaxlib_ver:
    print(f'Mismatch detected: jaxlib={jaxlib_ver}, jax_cuda12_plugin={plugin_ver}')
    repaired = False
    if WHEEL_DIR.exists():
        plugin_match = sorted(WHEEL_DIR.glob(f'jax_cuda12_plugin-{jaxlib_ver}-*.whl'))
        pjrt_match = sorted(WHEEL_DIR.glob(f'jax_cuda12_pjrt-{jaxlib_ver}-*.whl'))
        if plugin_match and pjrt_match:
            run_shell(
                f"{PYTHON} -m pip install --no-index --find-links {WHEEL_DIR} "
                f"--upgrade --force-reinstall --no-deps {plugin_match[-1]} {pjrt_match[-1]}"
            )
            repaired = True
    if not repaired:
        print('No matching plugin/pjrt wheel found; uninstalling stale CUDA plugin packages.')
        run_shell(f"{PYTHON} -m pip uninstall -y jax_cuda12_plugin jax_cuda12_pjrt", ok=(0, 1))

run_shell(f"{PYTHON} -m pip show jax jaxlib jax_cuda12_plugin jax_cuda12_pjrt flax optax datasets transformers", ok=(0, 1))
run_shell('nvidia-smi -L', ok=(0, 1))
run_shell('nvidia-smi', ok=(0, 1))

import jax

JAX_DEVICES = jax.devices()
DEVICE = 'gpu' if any(dev.platform == 'gpu' for dev in JAX_DEVICES) else 'cpu'
STRICT_JAX = True

RUN_ENV = os.environ.copy()
RUN_ENV.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
RUN_ENV.setdefault('XLA_PYTHON_CLIENT_MEM_FRACTION', '0.85')


GPU_STRICT_SUBPROCESS_OK = False
if DEVICE == 'gpu':
    gpu_probe = run_py(
        ['-c', "from iris.runtime import assert_jax_runtime; assert_jax_runtime(device='gpu', require_gpu=True); print('GPU_STRICT_OK')"],
        check=False,
        env=RUN_ENV,
    )
    GPU_STRICT_SUBPROCESS_OK = gpu_probe.returncode == 0
    if not GPU_STRICT_SUBPROCESS_OK:
        print('Subprocess strict GPU probe failed. Forcing DEVICE=cpu and STRICT_JAX=False for notebook stability.')
        DEVICE = 'cpu'
        STRICT_JAX = False


print('jax version:', jax.__version__)
print('jaxlib version:', pkg_version('jaxlib'))
print('jax_cuda12_plugin version:', pkg_version('jax_cuda12_plugin'))
print('jax_cuda12_pjrt version:', pkg_version('jax_cuda12_pjrt'))
print('jax devices:', JAX_DEVICES)
print('selected DEVICE:', DEVICE)
print('STRICT_JAX:', STRICT_JAX)
print('GPU_STRICT_SUBPROCESS_OK:', GPU_STRICT_SUBPROCESS_OK)


In [ ]:
PHASE = 'E'
BASELINE_ID = 'phase-e-v1'
TOLERANCE_PROFILE_ID = 'phase-e-default'

MAX_REASONING_CYCLES = 1
TERMINATION_THRESHOLD = 0.5
SEED = 17

TOTAL_TARGET_TOKENS = 100_000_000
HALF_TARGET_TOKENS = 50_000_000
TOKENS_PER_MICRO_STEP = 100_000
MICRO_STEPS = 10
SEGMENTS_PER_HALF = 50
REQUIRE_GPU_FOR_100M = True

if HALF_TARGET_TOKENS != TOKENS_PER_MICRO_STEP * MICRO_STEPS * SEGMENTS_PER_HALF:
    raise RuntimeError('Invalid token split: HALF_TARGET_TOKENS does not match segments/micro-steps/tokens config.')

MODEL_DIR = ARTIFACT_ROOT / 'model_run'
STREAMING_SMOKE_DIR = ARTIFACT_ROOT / 'streaming_smoke'
PHASE_ROOT = ARTIFACT_ROOT / 'phase_root'
ARC_PROBE_DIR = ARTIFACT_ROOT / 'arc_probe'

BASELINE_DIR = ARTIFACT_ROOT / 'baseline_phase_e_v1'
REPORT_FREEZE_DIR = ARTIFACT_ROOT / 'report_freeze'
REPORT_DIFF_DIR = ARTIFACT_ROOT / 'report_diff'

PINNED_MANIFEST = ARTIFACT_ROOT / 'runtime_lock_manifest_pinned.json'

H100_UNINTERRUPTED = ARTIFACT_ROOT / 'toy_train_gpu_h100'
H100_EXECUTE = ARTIFACT_ROOT / 'toy_resume_h100'
H100_PRE_COMMIT = ARTIFACT_ROOT / 'toy_resume_h100_pre_commit'
H100_POST_COMMIT = ARTIFACT_ROOT / 'toy_resume_h100_post_commit'

LOCAL_S8_UNINTERRUPTED = PHASE_ROOT / 's8_uninterrupted'
LOCAL_S8_EXECUTE = PHASE_ROOT / 's8_execute'
LOCAL_S8_PRE_COMMIT = PHASE_ROOT / 's8_pre_commit'
LOCAL_S8_POST_COMMIT = PHASE_ROOT / 's8_post_commit'

def strict_jax_flags() -> list[str]:
    return ['--device', DEVICE] + (['--strict-jax'] if STRICT_JAX else ['--no-strict-jax'])

print('Phase:', PHASE)
print('Target tokens:', TOTAL_TARGET_TOKENS, '(50/50 split)')
print('Half target tokens:', HALF_TARGET_TOKENS)
print('Segments per half:', SEGMENTS_PER_HALF)
print('Micro steps:', MICRO_STEPS)
print('Tokens per micro step:', TOKENS_PER_MICRO_STEP)


In [ ]:
runtime_lock_seed_dir = ARTIFACT_ROOT / 'runtime_lock_seed'
shutil.rmtree(runtime_lock_seed_dir, ignore_errors=True)

run_py(
    [
        'scripts/train_toy.py',
        '--output-dir', str(runtime_lock_seed_dir),
        '--segments', '0',
        '--phase', PHASE,
    ] + strict_jax_flags(),
    env=RUN_ENV,
)

seed_manifest = runtime_lock_seed_dir / 'runtime_lock_manifest.json'
if not seed_manifest.exists():
    raise FileNotFoundError(f'Runtime lock seed manifest missing: {seed_manifest}')

PINNED_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(seed_manifest, PINNED_MANIFEST)

print('Pinned runtime manifest:', PINNED_MANIFEST)


In [ ]:
if STREAMING_MODE == 'local_snapshot' and SNAPSHOT_ROOT is None:
    raise RuntimeError('STREAMING_MODE is local_snapshot but SNAPSHOT_ROOT is None.')
if not str(TOKENIZER_ID_OR_PATH).strip():
    raise RuntimeError('TOKENIZER_ID_OR_PATH is empty.')


SMOKE_EFFECTIVE_DEVICE = DEVICE
SMOKE_EFFECTIVE_STRICT_JAX = STRICT_JAX


def smoke_flags(device: str, strict_jax: bool) -> list[str]:
    return ['--device', str(device)] + (['--strict-jax'] if strict_jax else ['--no-strict-jax'])


def build_smoke_args(mode: str, snapshot_root: Path | None) -> list[str]:
    args = [
        'scripts/train_toy.py',
        '--output-dir', str(STREAMING_SMOKE_DIR),
        '--segments', '1',
        '--micro-steps', '4',
        '--tokens-per-micro-step', '4096',
        '--phase', PHASE,
        '--data-source', 'pure_lm_streaming',
        '--streaming-mode', str(mode),
        '--tokenizer-id-or-path', str(TOKENIZER_ID_OR_PATH),
        '--runtime-lock-manifest', str(PINNED_MANIFEST),
        '--resume-path-id', 'uninterrupted',
    ]
    if snapshot_root is not None:
        args.extend(['--snapshot-root', str(snapshot_root)])
    return args


def run_smoke_attempt(
    *,
    label: str,
    mode: str,
    snapshot_root: Path | None,
    device: str,
    strict_jax: bool,
    attempts: list[dict],
):
    shutil.rmtree(STREAMING_SMOKE_DIR, ignore_errors=True)
    args = build_smoke_args(mode, snapshot_root)
    proc = run_py(args + smoke_flags(device, strict_jax), check=False, env=RUN_ENV)
    attempts.append(
        {
            'label': label,
            'mode': mode,
            'snapshot_root': str(snapshot_root) if snapshot_root is not None else '',
            'device': device,
            'strict_jax': bool(strict_jax),
            'rc': int(proc.returncode),
        }
    )
    return proc


attempts: list[dict] = []
smoke_proc = run_smoke_attempt(
    label='primary',
    mode=STREAMING_MODE,
    snapshot_root=SNAPSHOT_ROOT,
    device=SMOKE_EFFECTIVE_DEVICE,
    strict_jax=SMOKE_EFFECTIVE_STRICT_JAX,
    attempts=attempts,
)

if smoke_proc.returncode != 0 and STREAMING_MODE == 'hf_online' and ENABLE_BOOTSTRAP_LOCAL_SNAPSHOT_FALLBACK:
    print('HF online streaming smoke failed. Falling back to bootstrap local snapshot...')
    SNAPSHOT_ROOT = bootstrap_local_snapshot(BOOTSTRAP_SNAPSHOT_ROOT)
    STREAMING_MODE = 'local_snapshot'
    smoke_proc = run_smoke_attempt(
        label='bootstrap_local_snapshot',
        mode=STREAMING_MODE,
        snapshot_root=SNAPSHOT_ROOT,
        device=SMOKE_EFFECTIVE_DEVICE,
        strict_jax=SMOKE_EFFECTIVE_STRICT_JAX,
        attempts=attempts,
    )

if smoke_proc.returncode != 0 and ALLOW_SMOKE_CPU_FALLBACK and SMOKE_EFFECTIVE_DEVICE == 'gpu':
    log_lower = (smoke_proc.stdout or '').lower()
    gpu_runtime_error = (
        'strict jax runtime requested gpu' in log_lower
        or 'no jax gpu device is available' in log_lower
        or 'jax plugin configuration error' in log_lower
    )
    if gpu_runtime_error:
        print('GPU strict runtime failed in smoke. Retrying smoke on CPU without strict-jax for data-path diagnosis...')
        smoke_proc = run_smoke_attempt(
            label='cpu_no_strict_fallback',
            mode=STREAMING_MODE,
            snapshot_root=SNAPSHOT_ROOT,
            device='cpu',
            strict_jax=False,
            attempts=attempts,
        )
        if smoke_proc.returncode == 0:
            SMOKE_EFFECTIVE_DEVICE = 'cpu'
            SMOKE_EFFECTIVE_STRICT_JAX = False

if smoke_proc.returncode != 0:
    print('Streaming smoke attempts summary:')
    for attempt in attempts:
        print(
            f"  - {attempt['label']}: rc={attempt['rc']} mode={attempt['mode']} "
            f"device={attempt['device']} strict={attempt['strict_jax']} snapshot={attempt['snapshot_root']}"
        )
    print('\nLast smoke output tail (120 lines):')
    tail_lines = (smoke_proc.stdout or '').splitlines()[-120:]
    print('\n'.join(tail_lines))
    raise RuntimeError(
        f'Streaming smoke failed after fallback attempts. rc={smoke_proc.returncode}. '
        'See attempt summary and output tail printed above.'
    )

smoke_metrics = read_jsonl(STREAMING_SMOKE_DIR / 'metrics.jsonl')
if not smoke_metrics:
    raise RuntimeError('Streaming smoke produced no metrics rows.')

smoke_last = smoke_metrics[-1]
effective_mode = str(smoke_last.get('data.streaming_mode_effective', ''))
if effective_mode != STREAMING_MODE:
    raise RuntimeError(f"Expected streaming mode '{STREAMING_MODE}', got '{effective_mode}'.")

ledger = smoke_last.get('data.token_ledger', {})
total_tokens = int((ledger or {}).get('total_tokens', 0))
if total_tokens <= 0:
    raise RuntimeError('Streaming smoke token ledger total_tokens must be > 0.')

by_source = dict((ledger or {}).get('by_source', {}))
non_synthetic_sources = [sid for sid in by_source.keys() if sid != 'synthetic_ir_aligned']
if not non_synthetic_sources:
    raise RuntimeError('Streaming smoke must include at least one non-synthetic source in token ledger.')

print('Streaming smoke PASS')
print('data.streaming_mode_effective =', effective_mode)
print('token ledger total_tokens =', total_tokens)
print('non-synthetic sources =', non_synthetic_sources)
print('effective STREAMING_MODE for following cells =', STREAMING_MODE)
print('effective SNAPSHOT_ROOT for following cells =', SNAPSHOT_ROOT)
print('smoke effective flags =', smoke_flags(SMOKE_EFFECTIVE_DEVICE, SMOKE_EFFECTIVE_STRICT_JAX))
print('train effective flags =', strict_jax_flags())
if REQUIRE_GPU_FOR_100M and DEVICE != 'gpu':
    print('WARNING: REQUIRE_GPU_FOR_100M=True but DEVICE is not gpu. Cell 6 will stop before 100M training.')


In [ ]:
if REQUIRE_GPU_FOR_100M and DEVICE != 'gpu':
    raise RuntimeError(
        'REQUIRE_GPU_FOR_100M=True but DEVICE is not gpu. Resolve JAX GPU runtime first, '
        'or set REQUIRE_GPU_FOR_100M=False for debug-only CPU runs.'
    )

if SNAPSHOT_ROOT is not None and Path(SNAPSHOT_ROOT) == BOOTSTRAP_SNAPSHOT_ROOT:
    print('Refreshing bootstrap local snapshot before 50M run...')
    SNAPSHOT_ROOT = bootstrap_local_snapshot(BOOTSTRAP_SNAPSHOT_ROOT)

shutil.rmtree(MODEL_DIR, ignore_errors=True)

train_half_args = [
    'scripts/train_pretrain.py',
    '--output-dir', str(MODEL_DIR),
    '--run-id', 'phase-e-kaggle-100m',
    '--segments', str(SEGMENTS_PER_HALF),
    '--micro-steps', str(MICRO_STEPS),
    '--tokens-per-micro-step', str(TOKENS_PER_MICRO_STEP),
    '--data-source', 'hybrid_mixture',
    '--hybrid-pure-ratio', '0.9',
    '--streaming-mode', STREAMING_MODE,
    '--tokenizer-id-or-path', str(TOKENIZER_ID_OR_PATH),
    '--baseline-id', BASELINE_ID,
    '--tolerance-profile-id', TOLERANCE_PROFILE_ID,
    '--runtime-lock-manifest', str(PINNED_MANIFEST),
    '--resume-path-id', 'uninterrupted',
]
if SNAPSHOT_ROOT is not None:
    train_half_args.extend(['--snapshot-root', str(SNAPSHOT_ROOT)])

train_half_args = train_half_args + strict_jax_flags()
run_py(train_half_args, env=RUN_ENV)

model_metrics_path = MODEL_DIR / 'metrics.jsonl'
model_journal_path = MODEL_DIR / 'segment_journal.jsonl'
metrics_after_half1 = read_jsonl(model_metrics_path)
journal_after_half1 = read_jsonl(model_journal_path)

if not metrics_after_half1:
    raise RuntimeError('First 50M run produced no metrics rows.')

half1_metrics_line_count = len(metrics_after_half1)
half1_last_segment = int(metrics_after_half1[-1].get('segment_id', -1))

print('Half-1 complete')
print('half1_metrics_line_count =', half1_metrics_line_count)
print('half1_last_segment =', half1_last_segment)
print('journal events after half1 =', len(journal_after_half1))


In [ ]:
from collections import Counter

run_py(train_half_args, env=RUN_ENV)

metrics_all = read_jsonl(MODEL_DIR / 'metrics.jsonl')
journal_all = read_jsonl(MODEL_DIR / 'segment_journal.jsonl')

applied_events = [row for row in journal_all if str(row.get('status', '')).upper() == 'APPLIED']
applied_segments = [int(row.get('segment_id', -1)) for row in applied_events]
expected_segments = list(range(SEGMENTS_PER_HALF * 2))

if sorted(set(applied_segments)) != expected_segments:
    raise RuntimeError('APPLIED segments do not cover 0..99 exactly.')

counts = Counter(applied_segments)
repeated = {seg: count for seg, count in counts.items() if count != 1}
if repeated:
    raise RuntimeError(f'Exactly-once violation detected for APPLIED segments: {repeated}')

if len(metrics_all) < half1_metrics_line_count:
    raise RuntimeError('metrics.jsonl row count is smaller than first-half snapshot.')

metrics_half1 = metrics_all[:half1_metrics_line_count]
metrics_half2 = metrics_all[half1_metrics_line_count:]
if len(metrics_half2) != SEGMENTS_PER_HALF:
    raise RuntimeError(
        f'Expected {SEGMENTS_PER_HALF} metrics rows for half-2, got {len(metrics_half2)}.'
    )

def ledger_tokens(metric_row: dict) -> int:
    return int(((metric_row.get('data.token_ledger') or {}).get('total_tokens')) or 0)

half1_tokens = ledger_tokens(metrics_half1[-1]) if metrics_half1 else 0
half2_tokens = ledger_tokens(metrics_half2[-1]) if metrics_half2 else 0

print('50/50 resume check PASS')
print('APPLIED segments range: 0..', max(applied_segments) if applied_segments else -1)
print('half1 token ledger total_tokens =', half1_tokens)
print('half2 token ledger total_tokens =', half2_tokens)
print('expected half target tokens =', HALF_TARGET_TOKENS)


In [ ]:
weights_dir = ARTIFACT_ROOT / 'weights'
weights_dir.mkdir(parents=True, exist_ok=True)

journal_rows = read_jsonl(MODEL_DIR / 'segment_journal.jsonl')
applied_rows = [row for row in journal_rows if str(row.get('status', '')).upper() == 'APPLIED']
if not applied_rows:
    raise RuntimeError('No APPLIED rows found for weight export.')

latest_applied = max(applied_rows, key=lambda row: int(row.get('segment_id', -1)))
checkpoint_ref = Path(str(latest_applied.get('checkpoint_ref', '')))
if not checkpoint_ref.exists():
    checkpoint_ref = MODEL_DIR / checkpoint_ref
if not checkpoint_ref.exists():
    raise FileNotFoundError(f'Latest checkpoint_ref does not exist: {checkpoint_ref}')

exported_checkpoint = weights_dir / 'phase_e_100m_checkpoint_latest.json'
shutil.copy2(checkpoint_ref, exported_checkpoint)

checkpoint_payload = json.loads(exported_checkpoint.read_text(encoding='utf-8-sig'))
model_state = checkpoint_payload.get('model_state')
if not isinstance(model_state, dict):
    raise RuntimeError('checkpoint payload missing model_state object.')

exported_model_state = weights_dir / 'phase_e_100m_model_state_latest.json'
exported_model_state.write_text(
    json.dumps(model_state, sort_keys=True, indent=2),
    encoding='utf-8',
)

print('weights exported:')
print('-', exported_checkpoint, 'bytes=', exported_checkpoint.stat().st_size)
print('-', exported_model_state, 'bytes=', exported_model_state.stat().st_size)


In [ ]:
def run_uninterrupted(output_dir: Path):
    shutil.rmtree(output_dir, ignore_errors=True)
    run_py(
        [
            'scripts/train_toy.py',
            '--output-dir', str(output_dir),
            '--segments', '1',
            '--phase', PHASE,
            '--resume-path-id', 'uninterrupted',
            '--runtime-lock-manifest', str(PINNED_MANIFEST),
        ] + strict_jax_flags(),
        env=RUN_ENV,
    )

def run_crash_resume(output_dir: Path, crash_point: str, resume_path_id: str):
    shutil.rmtree(output_dir, ignore_errors=True)
    run_py(
        [
            'scripts/train_toy.py',
            '--output-dir', str(output_dir),
            '--segments', '1',
            '--phase', PHASE,
            '--crash-point', crash_point,
            '--crash-segment', '0',
            '--runtime-lock-manifest', str(PINNED_MANIFEST),
        ] + strict_jax_flags(),
        ok=(0, 2),
        check=True,
        env=RUN_ENV,
    )
    run_py(
        [
            'scripts/train_toy.py',
            '--output-dir', str(output_dir),
            '--segments', '1',
            '--phase', PHASE,
            '--resume-path-id', resume_path_id,
            '--runtime-lock-manifest', str(PINNED_MANIFEST),
        ] + strict_jax_flags(),
        env=RUN_ENV,
    )

for path in [
    LOCAL_S8_UNINTERRUPTED,
    LOCAL_S8_EXECUTE,
    LOCAL_S8_PRE_COMMIT,
    LOCAL_S8_POST_COMMIT,
    H100_UNINTERRUPTED,
    H100_EXECUTE,
    H100_PRE_COMMIT,
    H100_POST_COMMIT,
]:
    shutil.rmtree(path, ignore_errors=True)

run_uninterrupted(LOCAL_S8_UNINTERRUPTED)
run_crash_resume(LOCAL_S8_EXECUTE, 'execute', 'execute_crash')
run_crash_resume(LOCAL_S8_PRE_COMMIT, 'pre_commit', 'pre_commit_crash')
run_crash_resume(LOCAL_S8_POST_COMMIT, 'post_commit', 'post_commit_crash')

run_uninterrupted(H100_UNINTERRUPTED)
run_crash_resume(H100_EXECUTE, 'execute', 'execute_crash')
run_crash_resume(H100_PRE_COMMIT, 'pre_commit', 'pre_commit_crash')
run_crash_resume(H100_POST_COMMIT, 'post_commit', 'post_commit_crash')

required_files = ['runtime_lock_manifest.json', 'segment_journal.jsonl', 'metrics.jsonl']
for run_dir in [
    LOCAL_S8_UNINTERRUPTED,
    LOCAL_S8_EXECUTE,
    LOCAL_S8_PRE_COMMIT,
    LOCAL_S8_POST_COMMIT,
    H100_UNINTERRUPTED,
    H100_EXECUTE,
    H100_PRE_COMMIT,
    H100_POST_COMMIT,
]:
    for name in required_files:
        candidate = run_dir / name
        if not candidate.exists():
            raise FileNotFoundError(f'Missing required artifact: {candidate}')
    checkpoint_files = sorted((run_dir / 'checkpoints').glob('segment_*.json'))
    if not checkpoint_files:
        raise RuntimeError(f'No checkpoint artifacts found under: {run_dir / "checkpoints"}')

print('S8 path rebuild PASS')
print('Local S8 root:', PHASE_ROOT)
print('H100 packet paths root:', ARTIFACT_ROOT)


In [ ]:
def find_arc_tasks_dir() -> Path | None:
    candidates: list[Path] = [
        ROOT / 'tools/arc-agi-benchmarking/data/sample/tasks',
        ROOT / 'tools/arc-agi-benchmarking/data/tasks',
        Path('/kaggle/input/arc-agi-benchmarking/data/sample/tasks'),
        Path('/kaggle/input/arc-agi-benchmarking/data/tasks'),
    ]

    input_root = Path('/kaggle/input')
    if input_root.exists():
        for pattern in ('*/data/sample/tasks', '*/data/tasks', '*/tasks'):
            for hit in input_root.glob(pattern):
                candidates.append(hit)
        for hit in input_root.rglob('data/sample/tasks'):
            candidates.append(hit)

    seen: set[str] = set()
    for candidate in candidates:
        key = str(candidate.resolve()) if candidate.exists() else str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if candidate.exists() and candidate.is_dir() and any(candidate.glob('*.json')):
            return candidate
    return None


def bootstrap_arc_tasks(tasks_dir: Path) -> Path:
    tasks_dir.mkdir(parents=True, exist_ok=True)
    bootstrap_tasks = {
        'bootstrap_identity': {
            'train': [
                {'input': [[0, 1], [2, 3]], 'output': [[0, 1], [2, 3]]},
                {'input': [[4, 4], [4, 4]], 'output': [[4, 4], [4, 4]]},
            ],
            'test': [
                {'input': [[5, 6], [7, 8]], 'output': [[5, 6], [7, 8]]},
            ],
        },
        'bootstrap_fill': {
            'train': [
                {'input': [[0, 0], [0, 0]], 'output': [[9, 9], [9, 9]]},
            ],
            'test': [
                {'input': [[1, 1], [1, 1]], 'output': [[9, 9], [9, 9]]},
            ],
        },
    }
    for task_id, payload in bootstrap_tasks.items():
        (tasks_dir / f'{task_id}.json').write_text(
            json.dumps(payload, sort_keys=True, indent=2),
            encoding='utf-8',
        )
    return tasks_dir


resolved_tasks_dir = find_arc_tasks_dir()
if resolved_tasks_dir is None:
    resolved_tasks_dir = bootstrap_arc_tasks(ARC_PROBE_DIR / 'bootstrap_tasks')
    print('ARC tasks directory not found; using bootstrap tasks:', resolved_tasks_dir)
else:
    print('Using ARC tasks directory:', resolved_tasks_dir)

probe_proc = run_py(
    [
        'scripts/run_arc_benchmark_probe.py',
        '--model-run-dir', str(MODEL_DIR),
        '--tasks-dir', str(resolved_tasks_dir),
        '--output-dir', str(ARC_PROBE_DIR),
        '--device', 'cpu',
        '--no-hard-fail',
    ],
    env=RUN_ENV,
    check=False,
)
if probe_proc.returncode != 0:
    raise RuntimeError(f'ARC benchmark probe failed (rc={probe_proc.returncode}). Check printed output above.')

arc_probe_path = ARC_PROBE_DIR / 'arc_benchmark_probe.json'
if not arc_probe_path.exists():
    raise FileNotFoundError(f'ARC probe artifact missing: {arc_probe_path}')

arc_probe = json.loads(arc_probe_path.read_text(encoding='utf-8-sig'))
print('arc_probe_path =', arc_probe_path)
print('tasks_dir_used =', resolved_tasks_dir)
print('status =', arc_probe.get('status'))
print('baseline_non_regression =', arc_probe.get('baseline_non_regression'))
print('block_reasons =', arc_probe.get('block_reasons', []))


In [ ]:
BASELINE_REQUIRED_FILES = (
    'failure_profile_diff.json',
    'concept_breakdown.json',
    'paired_representation_diff.json',
)


def baseline_artifacts_ready(base_dir: Path) -> bool:
    return base_dir.exists() and all((base_dir / name).exists() for name in BASELINE_REQUIRED_FILES)


shutil.rmtree(BASELINE_DIR, ignore_errors=True)
shutil.rmtree(REPORT_FREEZE_DIR, ignore_errors=True)

phase_e_base_args = [
    'scripts/phase_e_gate.py',
    '--phase', PHASE,
    '--model-run-dir', str(MODEL_DIR),
    '--phase-root', str(PHASE_ROOT),
    '--baseline-report-dir', str(BASELINE_DIR),
    '--conceptarc-corpus', 'tools/ConceptARC/corpus',
    '--rearc-tasks', 'data/arc/re_arc/tasks',
    '--arc-benchmark-probe', str(ARC_PROBE_DIR / 'arc_benchmark_probe.json'),
    '--pairing-policy', 'adjacent',
    '--baseline-id', BASELINE_ID,
    '--tolerance-profile-id', TOLERANCE_PROFILE_ID,
    '--device', DEVICE,
    '--max-reasoning-cycles', str(MAX_REASONING_CYCLES),
    '--termination-threshold', str(TERMINATION_THRESHOLD),
    '--seed', str(SEED),
    '--h100-uninterrupted', str(H100_UNINTERRUPTED),
    '--h100-execute', str(H100_EXECUTE),
    '--h100-pre-commit', str(H100_PRE_COMMIT),
    '--h100-post-commit', str(H100_POST_COMMIT),
]

freeze_proc = run_py(
    phase_e_base_args + [
        '--output-dir', str(REPORT_FREEZE_DIR),
        '--freeze-baseline',
        '--strict-suite-exec',
        '--no-reuse-existing-suite-artifacts',
    ],
    check=False,
    env=RUN_ENV,
)
print('phase_e_freeze_rc =', freeze_proc.returncode)
print('baseline_artifacts_ready =', baseline_artifacts_ready(BASELINE_DIR))
print('baseline_dir =', BASELINE_DIR)
print('baseline_files =', sorted(path.name for path in BASELINE_DIR.glob('*.json')) if BASELINE_DIR.exists() else [])

if not baseline_artifacts_ready(BASELINE_DIR):
    raise RuntimeError(
        'Freeze baseline did not produce required baseline artifacts. '
        'Fix freeze run before executing diff.'
    )


In [ ]:
shutil.rmtree(REPORT_DIFF_DIR, ignore_errors=True)

if 'phase_e_base_args' not in globals():
    phase_e_base_args = [
        'scripts/phase_e_gate.py',
        '--phase', PHASE,
        '--model-run-dir', str(MODEL_DIR),
        '--phase-root', str(PHASE_ROOT),
        '--baseline-report-dir', str(BASELINE_DIR),
        '--conceptarc-corpus', 'tools/ConceptARC/corpus',
        '--rearc-tasks', 'data/arc/re_arc/tasks',
        '--arc-benchmark-probe', str(ARC_PROBE_DIR / 'arc_benchmark_probe.json'),
        '--pairing-policy', 'adjacent',
        '--baseline-id', BASELINE_ID,
        '--tolerance-profile-id', TOLERANCE_PROFILE_ID,
        '--device', DEVICE,
        '--max-reasoning-cycles', str(MAX_REASONING_CYCLES),
        '--termination-threshold', str(TERMINATION_THRESHOLD),
        '--seed', str(SEED),
        '--h100-uninterrupted', str(H100_UNINTERRUPTED),
        '--h100-execute', str(H100_EXECUTE),
        '--h100-pre-commit', str(H100_PRE_COMMIT),
        '--h100-post-commit', str(H100_POST_COMMIT),
    ]

if 'baseline_artifacts_ready' not in globals():
    BASELINE_REQUIRED_FILES = (
        'failure_profile_diff.json',
        'concept_breakdown.json',
        'paired_representation_diff.json',
    )

    def baseline_artifacts_ready(base_dir: Path) -> bool:
        return base_dir.exists() and all((base_dir / name).exists() for name in BASELINE_REQUIRED_FILES)

if not baseline_artifacts_ready(BASELINE_DIR):
    print('Baseline artifacts missing; running freeze baseline automatically before diff...')
    shutil.rmtree(REPORT_FREEZE_DIR, ignore_errors=True)
    freeze_proc = run_py(
        phase_e_base_args + [
            '--output-dir', str(REPORT_FREEZE_DIR),
            '--freeze-baseline',
            '--strict-suite-exec',
            '--no-reuse-existing-suite-artifacts',
        ],
        check=False,
        env=RUN_ENV,
    )
    print('phase_e_freeze_rc =', freeze_proc.returncode)

if not baseline_artifacts_ready(BASELINE_DIR):
    raise RuntimeError('Baseline artifacts are still missing after freeze attempt; diff gate cannot proceed.')

diff_proc = run_py(
    phase_e_base_args + [
        '--output-dir', str(REPORT_DIFF_DIR),
        '--strict-suite-exec',
        '--no-reuse-existing-suite-artifacts',
    ],
    check=False,
    env=RUN_ENV,
)
print('phase_e_diff_rc =', diff_proc.returncode)

summary_path = REPORT_DIFF_DIR / 'summary_report.json'
if not summary_path.exists():
    summary_path = REPORT_FREEZE_DIR / 'summary_report.json'
if not summary_path.exists():
    print('REPORT_DIFF_DIR files =', sorted(path.name for path in REPORT_DIFF_DIR.glob('*')) if REPORT_DIFF_DIR.exists() else [])
    print('REPORT_FREEZE_DIR files =', sorted(path.name for path in REPORT_FREEZE_DIR.glob('*')) if REPORT_FREEZE_DIR.exists() else [])
    raise FileNotFoundError('summary_report.json not found in diff or freeze outputs.')

summary = json.loads(summary_path.read_text(encoding='utf-8-sig'))
regression_status = str(summary.get('regression.status', 'FAIL')).upper()

print('summary_path =', summary_path)
print('regression.status =', regression_status)
print('suite_status =')
print(json.dumps(summary.get('suite_status', {}), indent=2, ensure_ascii=False))
print('regression.violations =')
print(json.dumps(summary.get('regression.violations', []), indent=2, ensure_ascii=False))
print('completion_checklist =')
print(json.dumps(summary.get('completion_checklist', {}), indent=2, ensure_ascii=False))

if REQUIRE_GATE_PASS and regression_status != 'PASS':
    raise RuntimeError('Phase E diff gate is not PASS and REQUIRE_GATE_PASS=True.')


In [ ]:
zip_base = Path('/kaggle/working/iris_phase_e_pretrain_streaming_outputs')
zip_file = zip_base.with_suffix('.zip')
if zip_file.exists():
    zip_file.unlink()

archive_path = shutil.make_archive(str(zip_base), 'zip', root_dir=str(ARTIFACT_ROOT))
print('compressed artifact root:', ARTIFACT_ROOT)
print('zip output:', archive_path)
print('size bytes:', Path(archive_path).stat().st_size)
